# Nonlinear Mesh Parametrization using MeshFEM's new `MeshEnergy` class

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark
import energy

m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
# Initialization
uv.setVars(parametrization.lscm(m).ravel())

In [ ]:
# Select parametrization energy
# (any 2x2 F-based energy will work, but the associated MeshEnergy instantiation must be bound by adding a line in `mesh_energy.cc`
e = energy.NeoHookeanYoungPoissonAutoProjected(2, 1.0, 0.3)
e = energy.NeoHookeanYoungPoisson(2, 1.0, 0.3)

In [ ]:
# Construct parametrization energy and problem
param = mesh_energy.Parametrization(m, uv, e)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-8
opt = prob.optimizer()

In [ ]:
benchmark.reset()
opt.optimize()
benchmark.report()

In [ ]:
v.update()